# Hosting Strands Agents with Amazon Bedrock models in Amazon Bedrock AgentCore Runtime

## Overview

In this tutorial we will learn how to host your existing agent, using Amazon Bedrock AgentCore Runtime. We will provide examples using Amazon Bedrock models and non-Bedrock models such as Azure OpenAI and Gemini.


### Tutorial Details


| Information         | Details                                                                          |
|:--------------------|:---------------------------------------------------------------------------------|
| Tutorial type       | Conversational                                                                   |
| Agent type          | Single                                                                           |
| Agentic Framework   | Strands Agents                                                                   |
| LLM model           | Anthropic Claude Sonnet 3.7                                                        |
| Tutorial components | Hosting agent on AgentCore Runtime. Using Strands Agent and Amazon Bedrock Model |
| Tutorial vertical   | Cross-vertical                                                                   |
| Example complexity  | Easy                                                                             |
| SDK used            | Amazon BedrockAgentCore Python SDK and boto3                                     |

### Tutorial Architecture

In this tutorial we will describe how to deploy an existing agent to AgentCore runtime. 

For demonstration purposes, we will  use a Strands Agent using Amazon Bedrock models

In our example we will use a very simple agent with two tools: `get_weather` and `get_time`. 

<div style="text-align:left">
    <img src="images/architecture_runtime.png" width="50%"/>
</div>

### Tutorial Key Features

* Hosting Agents on Amazon Bedrock AgentCore Runtime
* Using Amazon Bedrock models
* Using Strands Agents


## Prerequisites

To execute this tutorial you will need:
* Python 3.10+
* AWS credentials
* Amazon Bedrock AgentCore SDK
* Strands Agents

In [ ]:
!pip install --force-reinstall -U -r requirements.txt --quiet

In [ ]:
!pip install strands-agents-tools uv boto3 bedrock-agentcore bedrock-agentcore-starter-toolkit

## Creating your agents and experimenting locally

Before we deploy our agents to AgentCore Runtime, let's develop and run them locally for experimentation purposes.

For production agentic applications we will need to decouple the agent creation process from the agent invocation one. With AgentCore Runtime, we will decorate the invocation part of our agent with the `@app.entrypoint` decorator and have it as the entry point for our runtime. Let's first look how each agent is developed during the experimentation phase.

The architecture here will look as following:

<div style="text-align:left">
    <img src="images/architecture_local.png" width="50%"/>
</div>

In [ ]:
%%writefile strands_claude.py
from strands import Agent, tool
from strands_tools import calculator # Import the calculator tool
import argparse
import json
from strands.models import BedrockModel
import boto3

@tool
def list_running_ec2_instances():
    """List running EC2 instances in us-east-1"""
    ec2 = boto3.client("ec2", region_name="us-east-1")
    response = ec2.describe_instances(Filters=[{"Name": "instance-state-name", "Values": ["running"]}])
    instances = []
    for reservation in response["Reservations"]:
        for instance in reservation["Instances"]:
            instances.append(instance["InstanceId"])
    return f"Running EC2 instances: {', '.join(instances)}" if instances else "No running instances found."

@tool
def query_config_aggregator(expression: str, aggregator_name: str = "TestName"):
    """Run a custom AWS Config aggregator query using select_aggregate_resource_config"""
    client = boto3.client("config", region_name="us-east-1")
    try:
        response = client.select_aggregate_resource_config(
            ConfigurationAggregatorName=aggregator_name,
            Expression=expression
        )
        results = response.get("Results", [])
        return f"Query returned {len(results)} results:\n" + "\n".join(results)
    except Exception as e:
        return f"Error running Config query: {str(e)}"

@tool
def list_config_resources(resource_type: str, aggregator_name: str = "TestName"):
    """List discovered resources of a given type using AWS Config aggregator"""
    client = boto3.client("config", region_name="us-east-1")
    try:
        response = client.list_aggregate_discovered_resources(
            ConfigurationAggregatorName=aggregator_name,
            ResourceType=resource_type
        )
        resources = response.get("ResourceIdentifiers", [])
        return f"Found {len(resources)} resources of type {resource_type}:\n" + "\n".join([r["ResourceId"] for r in resources])
    except Exception as e:
        return f"Error listing resources: {str(e)}"


model_id = "us.anthropic.claude-3-7-sonnet-20250219-v1:0"
model = BedrockModel(
    model_id=model_id,
)
agent = Agent(
    model=BedrockModel(model_id="us.anthropic.claude-3-5-sonnet-20240620-v1:0"),  # Update model ID if needed
    tools=[list_running_ec2_instances, query_config_aggregator, list_config_resources],
    system_prompt="You're a helpful assistant that can list EC2 instances, query AWS Config aggregators, and list Config resources."
)

def strands_agent_bedrock(payload):
    """
    Invoke the agent with a payload
    """
    user_input = payload.get("prompt")
    response = agent(user_input)
    return response.message['content'][0]['text']

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("payload", type=str)
    args = parser.parse_args()
    response = strands_agent_bedrock(json.loads(args.payload))

#### Invoking local agent

In [ ]:
!python strands_claude.py '{"prompt": "list running ec2 instances"}'

In [ ]:
!python strands_claude.py '{"prompt": "Can you list all the EC2 instances discovered in my AWS Config aggregator named TestName?"}'

In [ ]:
!python strands_claude.py '{"prompt": "un a query to find all non-compliant S3 buckets in the TestName config aggregator. Use a SQL expression to select resource ID and compliance status where the resource type is S3 bucket and compliance is NON_COMPLIANT."}'

## Preparing your agent for deployment on AgentCore Runtime

Let's now deploy our agents to AgentCore Runtime. To do so we need to:
* Import the Runtime App with `from bedrock_agentcore.runtime import BedrockAgentCoreApp`
* Initialize the App in our code with `app = BedrockAgentCoreApp()`
* Decorate the invocation function with the `@app.entrypoint` decorator
* Let AgentCoreRuntime control the running of the agent with `app.run()`

### Strands Agents with Amazon Bedrock model
Let's start with our Strands Agent using Amazon Bedrock model. All the others will work exactly the same.

In [ ]:
%%writefile strands_claude.py
from strands import Agent, tool
from strands_tools import calculator # Import the calculator tool
import argparse
import json
from bedrock_agentcore.runtime import BedrockAgentCoreApp
from strands.models import BedrockModel
import boto3

app = BedrockAgentCoreApp()

# Create a custom tool 
@tool
def list_running_ec2_instances():
    """List running EC2 instances in us-east-1"""
    ec2 = boto3.client("ec2", region_name="us-east-1")
    response = ec2.describe_instances(Filters=[{"Name": "instance-state-name", "Values": ["running"]}])
    instances = []
    for reservation in response["Reservations"]:
        for instance in reservation["Instances"]:
            instances.append(instance["InstanceId"])
    return f"Running EC2 instances: {', '.join(instances)}" if instances else "No running instances found."

@tool
def query_config_aggregator(expression: str, aggregator_name: str = "TestName"):
    """Run a custom AWS Config aggregator query using select_aggregate_resource_config"""
    client = boto3.client("config", region_name="us-east-1")
    try:
        response = client.select_aggregate_resource_config(
            ConfigurationAggregatorName=aggregator_name,
            Expression=expression
        )
        results = response.get("Results", [])
        return f"Query returned {len(results)} results:\n" + "\n".join(results)
    except Exception as e:
        return f"Error running Config query: {str(e)}"

@tool
def list_config_resources(resource_type: str, aggregator_name: str = "TestName"):
    """List discovered resources of a given type using AWS Config aggregator"""
    client = boto3.client("config", region_name="us-east-1")
    try:
        response = client.list_aggregate_discovered_resources(
            ConfigurationAggregatorName=aggregator_name,
            ResourceType=resource_type
        )
        resources = response.get("ResourceIdentifiers", [])
        return f"Found {len(resources)} resources of type {resource_type}:\n" + "\n".join([r["ResourceId"] for r in resources])
    except Exception as e:
        return f"Error listing resources: {str(e)}"


model_id = "us.anthropic.claude-3-7-sonnet-20250219-v1:0"
model = BedrockModel(
    model_id=model_id,
)
agent = Agent(
    model=model,
    tools=[calculator, list_config_resources, query_config_aggregator, list_running_ec2_instances],
    system_prompt="You are a helpful assistand that helps me with my AWS resources"
)

@app.entrypoint
def strands_agent_bedrock(payload):
    """
    Invoke the agent with a payload
    """
    user_input = payload.get("prompt")
    print("User input:", user_input)
    response = agent(user_input)
    return response.message['content'][0]['text']

if __name__ == "__main__":
    app.run()

In [ ]:
!echo -e "strands-agents-tools\nuv\nboto3\nbedrock-agentcore\nbedrock-agentcore-starter-toolkit" > requirements.txt

In [ ]:
!cat requirements.txt

## What happens behind the scenes?

When you use `BedrockAgentCoreApp`, it automatically:

* Creates an HTTP server that listens on the port 8080
* Implements the required `/invocations` endpoint for processing the agent's requirements
* Implements the `/ping` endpoint for health checks (very important for asynchronous agents)
* Handles proper content types and response formats
* Manages error handling according to the AWS standards

## Deploying the agent to AgentCore Runtime

The `CreateAgentRuntime` operation supports comprehensive configuration options, letting you specify container images, environment variables and encryption settings. You can also configure protocol settings (HTTP, MCP) and authorization mechanisms to control how your clients communicate with the agent. 

**Note:** Operations best practice is to package code as container and push to ECR using CI/CD pipelines and IaC

In this tutorial can will the Amazon Bedrock AgentCore Python SDK to easily package your artifacts and deploy them to AgentCore runtime.

### Configure AgentCore Runtime deployment

First we will use our starter toolkit to configure the AgentCore Runtime deployment with an entrypoint, the execution role we just created and a requirements file. We will also configure the starter kit to auto create the Amazon ECR repository on launch.

During the configure step, your docker file will be generated based on your application code

<div style="text-align:left">
    <img src="images/configure.png" width="60%"/>
</div>

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime
from boto3.session import Session
boto_session = Session()
region = boto_session.region_name

agentcore_runtime = Runtime()
agent_name = "strands_claude_getting_started_amazon_tools"
response = agentcore_runtime.configure(
    entrypoint="strands_claude.py",
    execution_role="arn:aws:iam::794038231401:role/AmazonBedrockAgentCoreSDKRuntime-us-east-1-4d0fb8fc25",
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    agent_name=agent_name,

)
print(response)

### Launching agent to AgentCore Runtime

Now that we've got a docker file, let's launch the agent to the AgentCore Runtime. This will create the Amazon ECR repository and the AgentCore Runtime

<div style="text-align:left">
    <img src="images/launch.png" width="75%"/>
</div>

In [ ]:
launch_result = agentcore_runtime.launch()

### Checking for the AgentCore Runtime Status
Now that we've deployed the AgentCore Runtime, let's check for it's deployment status

In [ ]:
import time
status_response = agentcore_runtime.status()
status = status_response.endpoint['status']
end_status = ['READY', 'CREATE_FAILED', 'DELETE_FAILED', 'UPDATE_FAILED']
while status not in end_status:
    time.sleep(10)
    status_response = agentcore_runtime.status()
    status = status_response.endpoint['status']
    print(status)
print(status)

### Invoking AgentCore Runtime

Finally, we can invoke our AgentCore Runtime with a payload

<div style="text-align:left">
    <img src="images/invoke.png" width=75%"/>
</div>

In [ ]:
invoke_response = agentcore_runtime.invoke({"prompt": "list ec2 instances"})
invoke_response

### Processing invocation results

We can now process our invocation results to include it in an application

In [ ]:
from IPython.display import Markdown, display
import json
response_text = json.loads(invoke_response['response'][0].decode("utf-8"))
display(Markdown(response_text))

### Invoking AgentCore Runtime with boto3

Now that your AgentCore Runtime was created you can invoke it with any AWS SDK. For instance, you can use the boto3 `invoke_agent_runtime` method for it.

In [ ]:
import boto3
agent_arn = launch_result.agent_arn
agentcore_client = boto3.client(
    'bedrock-agentcore',
    region_name=region
)

boto3_response = agentcore_client.invoke_agent_runtime(
    agentRuntimeArn=agent_arn,
    qualifier="DEFAULT",
    payload=json.dumps({"prompt": "What is 2+2?"})
)
if "text/event-stream" in boto3_response.get("contentType", ""):
    content = []
    for line in boto3_response["response"].iter_lines(chunk_size=1):
        if line:
            line = line.decode("utf-8")
            if line.startswith("data: "):
                line = line[6:]
                print(line)
                content.append(line)
    display(Markdown("\n".join(content)))
else:
    try:
        events = []
        for event in boto3_response.get("response", []):
            events.append(event)
    except Exception as e:
        events = [f"Error reading EventStream: {e}"]
    display(Markdown(json.loads(events[0].decode("utf-8"))))

## Cleanup (Optional)

Let's now clean up the AgentCore Runtime created

In [ ]:
launch_result.ecr_uri, launch_result.agent_id, launch_result.ecr_uri.split('/')[1]

In [ ]:
agentcore_control_client = boto3.client(
    'bedrock-agentcore-control',
    region_name=region
)
ecr_client = boto3.client(
    'ecr',
    region_name=region

)

runtime_delete_response = agentcore_control_client.delete_agent_runtime(
    agentRuntimeId=launch_result.agent_id,

)

response = ecr_client.delete_repository(
    repositoryName=launch_result.ecr_uri.split('/')[1],
    force=True
)

# Congratulations!